In [36]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.preprocessing import StandardScaler,Normalizer,OneHotEncoder,LabelEncoder,OrdinalEncoder,MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression,Lasso,Ridge,ElasticNet
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.neighbors import KNeighborsClassifier
from mlxtend.plotting import plot_decision_regions
from sklearn.ensemble import AdaBoostClassifier,AdaBoostRegressor,BaggingClassifier,GradientBoostingClassifier,HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error,mean_squared_error,mean_absolute_error,confusion_matrix,classification_report,accuracy_score,precision_score,r2_score
import xgboost as xgb

Data Collection and Analysis

In [37]:
df=pd.read_csv("/Users/prathamsharma/Desktop/Projects/Diabetes_Prediction using SVM/diabetes.csv")

In [38]:
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
36,11,138,76,0,0,33.2,0.420,35,0
691,13,158,114,0,0,42.3,0.257,44,1
279,2,108,62,10,278,25.3,0.881,22,0
391,5,166,76,0,0,45.7,0.340,27,1
708,9,164,78,0,0,32.8,0.148,45,1


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [40]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [41]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [42]:
df.shape

(768, 9)

In [43]:
df["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

0 marked as Non Diabetic Patients
1 marked as Diabetic Patients

Splitting the data for training and testing 

In [44]:
x=df.drop("Outcome",axis =1)
y=df["Outcome"]
print(x.shape)
print(y.shape)

(768, 8)
(768,)


In [45]:
x_train ,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(614, 8)
(154, 8)
(614,)
(154,)


Data Standardization

In [46]:
standard = StandardScaler()
x_train_scaled = pd.DataFrame(standard.fit_transform(x_train))
x_test_scaled = pd.DataFrame(standard.transform(x_test))

In [47]:
print(x_train_scaled.sample(5))
print(x_test_scaled.sample(5))

            0         1         2         3         4         5         6  \
318  0.356576 -0.821579  0.139061  0.771491 -0.730766  0.729057 -0.325510   
166 -0.549372 -1.265524  0.139061 -0.364621 -0.024767 -0.238485  0.210803   
521 -1.153338 -0.536185  0.356599 -1.311380 -0.730766 -1.727990  0.316854   
67  -0.247390  1.556702 -0.296015  1.023960  1.359364  0.321671 -0.367930   
74   2.470454  0.605390  0.682906  1.402664  2.288310  0.920019  0.153233   

            7  
318 -0.538444  
166 -0.707594  
521 -0.538444  
67  -0.284718  
74   2.083387  
            0         1         2         3         4         5         6  \
61  -1.153338 -0.472764  0.030292  1.023960  0.644075  0.945480  0.386544   
148  2.772437 -0.472764  0.139061  2.096954 -0.730766  0.589018 -0.907273   
130 -0.851355 -1.170393 -0.296015  0.140318  0.337522  0.627210 -0.019478   
4   -0.549372 -1.487497 -3.776623 -1.311380 -0.730766 -4.070459 -1.137554   
135 -1.153338 -0.694737 -3.776623 -1.311380 -0.730766 -0

MODEL TRAINING

In [48]:
svm = svm.SVC(kernel="linear")
svm.fit(x_train,y_train)
y_pred_test = svm.predict(x_test)
y_pred_train = svm.predict(x_train)
print("ACCURACY SCORE OF TRAINING DATA:",accuracy_score(y_train,y_pred_train))
print("ACCURACY SCORE OF TESTING DATA:",accuracy_score(y_test,y_pred_test))

ACCURACY SCORE OF TRAINING DATA: 0.7915309446254072
ACCURACY SCORE OF TESTING DATA: 0.7207792207792207


NO OVERFITTING DONE

MAKING A PRECTIVE MODEL

In [50]:
input_data = (11 ,138 , 76 , 0 , 0 , 33.2 , 0.420 , 35)

input_data_to_feed = np.asarray(input_data).reshape(1,-1)
# we are reshaping since the model is trained on 614,8 datapoints and now we are giving less data to the model so to mention the number of data points we are feeding to the model


input_data_to_feed_standard = standard.transform(input_data_to_feed)
#since we did standardize our data then thus we need to do for this input too
y_pred_input_data = svm.predict(input_data_to_feed_standard)
print(y_pred_input_data)

[0]


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
